# Model 1: Randomforest

#### Import necessary dependencies

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split

In [ ]:
# File path
final_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_full_cleaned_timestamped_encoded.parquet"

### 1. Load dataset via DuckDB to pandas (to avoid loading unnecessary columns)

In [ ]:
con = duckdb.connect()
# Select all columns except 'book_state' (target) and features (already encoded)
df = con.execute(f"SELECT * FROM '{final_file}'").fetchdf()

con.close()

print(f"Dataset shape: {df.shape}")
print(df.head())

### 2. Prepare features X and target y

In [ ]:
# Target column:
target = 'book_state'

# Check if 'book_state' exists in df (it should!)
if target not in df.columns:
    raise ValueError(f"Target column '{target}' not found in dataset.")

# Features - all numeric columns except 'book_state'
X = df.drop(columns=[target])
y = df[target]

### 3. Split the dataset (70% train, 15% val, 15% test)

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.1765, random_state=42,
                                                  stratify=y_trainval)
# 0.1765 ≈ 15/85 so final splits are ~70/15/15

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test.shape}")

### 4. Train Random Forest classifier

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training RandomForest prediction_models...")
clf.fit(X_train, y_train)

# 5. Evaluate on validation set

In [ ]:
y_val_pred = clf.predict(X_val)
print("Validation classification report:")
print(classification_report(y_val, y_val_pred))

# 6. Evaluate on test set

In [ ]:
y_test_pred = clf.predict(X_test)
print("Test classification report:")
print(classification_report(y_test, y_test_pred))

### 7. Metrics

In [ ]:
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, zero_division=0)
rec = recall_score(y_test, y_test_pred, zero_division=0)
f1 = f1_score(y_test, y_test_pred, zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix:")
print(cm)

# 7. Confusion Matrix Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pass (0)", "Fail (1)"], yticklabels=["Pass (0)", "Fail (1)"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()